# Notebook 05 — Model Evaluation & Selection

**Maternal & Infant Mortality Risk Prediction Using Data Mining Techniques**

> ⚠️ **SYNTHETIC — generated for academic demonstration. Not real clinical data.**

---

## Agenda
1. Comparison table: Accuracy, Precision, Recall, F1, ROC-AUC
2. Confusion matrices for every model
3. Overlaid ROC curves
4. Feature importance (Random Forest)
5. SHAP summary plot
6. 5-fold cross-validation (mean ± std)
7. Final model selection rationale
8. Save the winning pipeline

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

from src.evaluation import (
    compute_metrics,
    plot_confusion_matrices,
    plot_roc_curves,
    plot_feature_importance,
    plot_shap_summary,
    cross_validate_models,
    save_best_model,
)

FIGURES = os.path.join('..', 'reports', 'figures')
print('Imports OK')

In [ ]:
# Load cached models and data from Notebook 04
cache = joblib.load(os.path.join('..', 'models', 'all_trained_models.joblib'))

best_models    = cache['models']
preprocessor   = cache['preprocessor']
feature_names  = cache['feature_names']
X_train        = cache['X_train']
X_test         = cache['X_test']
y_train        = cache['y_train']
y_test         = cache['y_test']

print(f"Loaded {len(best_models)} models.")
print(f"Test set: {X_test.shape[0]} samples")

## 1. Metrics Comparison Table

In [ ]:
metrics_df = compute_metrics(best_models, X_test, y_test)
print("SYNTHETIC — Model Performance Comparison (sorted by Recall):")
metrics_df.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=0)

## 2. Confusion Matrices

In [ ]:
plot_confusion_matrices(best_models, X_test, y_test, save_dir=FIGURES)

## 3. ROC Curves

In [ ]:
plot_roc_curves(best_models, X_test, y_test, save_dir=FIGURES)

## 4. Feature Importance (Random Forest)

In [ ]:
rf_model = best_models['Random Forest']
plot_feature_importance(rf_model, feature_names, save_dir=FIGURES)

## 5. SHAP Summary Plot

In [ ]:
# SHAP explains WHY the model makes each prediction.
# We use the Random Forest model for SHAP analysis.
plot_shap_summary(rf_model, X_test, feature_names, save_dir=FIGURES)

## 6. 5-Fold Cross-Validation

In [ ]:
print("SYNTHETIC — 5-Fold Cross-Validation (Recall):")
cv_df = cross_validate_models(best_models, X_train, y_train, cv=5)
cv_df[['CV_Mean', 'CV_Std']].style.format('{:.4f}')

## 7. Final Model Selection — Why RECALL Matters Most

In maternal-infant risk prediction, the cost of a **false negative** (classifying
a HIGH-RISK pregnancy as LOW-RISK) is far greater than a false positive:

- **False negative** → A high-risk mother does NOT receive timely intervention →
  potential maternal/neonatal death.
- **False positive** → A low-risk mother receives extra monitoring → minor
  inconvenience but no clinical harm.

Therefore, we select the model with the **highest RECALL on the positive class
(HIGH-RISK = 1)**, not the one with the highest raw accuracy.

If a model achieves very high recall at the cost of moderate precision, that is
an acceptable trade-off in this healthcare context — it is better to over-screen
than to miss a genuinely at-risk patient.

In [ ]:
# Select best model by recall
best_model_name = metrics_df['Recall'].idxmax()
best_model = best_models[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Recall:   {metrics_df.loc[best_model_name, 'Recall']:.4f}")
print(f"   F1-Score: {metrics_df.loc[best_model_name, 'F1-Score']:.4f}")
print(f"   ROC-AUC:  {metrics_df.loc[best_model_name, 'ROC-AUC']:.4f}")
print(f"   Accuracy: {metrics_df.loc[best_model_name, 'Accuracy']:.4f}")

## 8. Save the Winning Model Pipeline

In [ ]:
# Save the best model + preprocessor for deployment in the Streamlit app
save_best_model(best_model, preprocessor)

# Also save the metrics table and feature names for the app
metrics_df.to_csv(os.path.join('..', 'models', 'metrics_comparison.csv'))
joblib.dump(feature_names, os.path.join('..', 'models', 'feature_names.joblib'))

print("\n✅ Best model pipeline saved to models/best_model_pipeline.joblib")
print("✅ Metrics table saved to models/metrics_comparison.csv")
print("✅ Feature names saved to models/feature_names.joblib")

## Summary

- All 7 classifiers evaluated on the held-out test set.
- Metrics comparison shows each model's Accuracy, Precision, Recall, F1 and ROC-AUC.
- Confusion matrices and ROC curves visualise each model's performance.
- Feature importance and SHAP explain which features drive predictions.
- The model with **highest recall on the HIGH-RISK class** is selected as the winner.
- The winning pipeline (model + preprocessor) is serialised for deployment.

**Next → Phase 7: Streamlit Application**